In [1]:
from pathlib import Path
import json, pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
P, M = ROOT / "data" / "processed", ROOT / "models"

train = pd.read_parquet(P / "train.parquet")
test  = pd.read_parquet(P / "test.parquet")
kaggle = pd.read_parquet(P / "kaggle_test.parquet")

with open(M / "binning_process.pkl", "rb") as f:
    bp = pickle.load(f)
with open(M / "lr_model.json") as f:
    lr = json.load(f)
gbm = lgb.Booster(model_file=str(M / "lgbm.txt"))

print(len(train), len(test), len(kaggle))


(CVXPY) Nov 10 11:09:59 AM: Encountered unexpected exception importing solver HIGHS:
ImportError('/home/fiza/Credit_Risk/.venv/lib/python3.12/site-packages/highspy/_core.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN5Highs13releaseMemoryEv')


215257 92254 48744


In [2]:
all_features = list(bp.variable_names)
coefs = pd.Series(lr["params"]).drop("const")

def scorecard_score(df):
    """Returns (score, probability of default) using the saved scorecard."""
    woe = bp.transform(df[all_features], metric="woe",
                       metric_missing="empirical", metric_special="empirical")
    W = pd.DataFrame(np.asarray(woe), columns=all_features, index=df.index)[lr["features"]]
    logit = lr["params"]["const"] + W @ coefs
    score = lr["offset"] - lr["factor"] * logit
    pd_ = 1 / (1 + np.exp(-logit))
    return score, pd_

gbm_features = gbm.feature_name()
def lgbm_predict(df):
    Xd = df[gbm_features].copy()
    for c in Xd.columns:
        if not pd.api.types.is_numeric_dtype(Xd[c]):
            Xd[c] = Xd[c].astype("category")
    return pd.Series(gbm.predict(Xd), index=df.index)


In [3]:
def gini_ks(y, risk):
    auc = roc_auc_score(y, risk)
    fpr, tpr, _ = roc_curve(y, risk)
    return round(2 * auc - 1, 3), round((tpr - fpr).max(), 3)

train_score, train_pd = scorecard_score(train)
test_score,  test_pd  = scorecard_score(test)

rows = []
for name, df, risk_sc, risk_gb in [("train", train, train_pd, lgbm_predict(train)),
                                    ("test",  test,  test_pd,  lgbm_predict(test))]:
    g, k = gini_ks(df["TARGET"], risk_sc); rows.append(("Scorecard", name, g, k))
    g, k = gini_ks(df["TARGET"], risk_gb); rows.append(("LightGBM",  name, g, k))

pd.DataFrame(rows, columns=["model", "data", "Gini", "KS"])


,model,data,Gini,KS
0,Scorecard,train,0.511,0.383
1,LightGBM,train,0.687,0.525
2,Scorecard,test,0.498,0.375
3,LightGBM,test,0.543,0.407


In [4]:
dec = pd.qcut(test_score, 10, labels=range(1, 11))     # 1 = lowest scores = riskiest

t = test.groupby(dec, observed=True)["TARGET"].agg(people="size", defaults="sum")
t["score_from"] = test_score.groupby(dec, observed=True).min().round()
t["score_to"]   = test_score.groupby(dec, observed=True).max().round()
t["actual_%"]    = (t["defaults"] / t["people"] * 100).round(1)
t["predicted_%"] = (test_pd.groupby(dec, observed=True).mean() * 100).round(1)
t["cum_defaults_caught_%"] = (t["defaults"].cumsum() / t["defaults"].sum() * 100).round(1)
t


,people,defaults,score_from,score_to,actual_%,predicted_%,cum_defaults_caught_%
1,9226,2412,458.0,531.0,26.1,26.8,32.4
2,9225,1379,531.0,544.0,14.9,14.8,50.9
3,9225,1003,544.0,554.0,10.9,10.5,64.4
4,9226,694,554.0,562.0,7.5,8.0,73.7
5,9225,549,562.0,569.0,6.0,6.2,81.1
6,9225,455,569.0,577.0,4.9,4.9,87.2
7,9226,335,577.0,584.0,3.6,3.8,91.7
8,9225,285,584.0,593.0,3.1,2.9,95.5
9,9225,203,593.0,604.0,2.2,2.1,98.2
10,9226,133,604.0,661.0,1.4,1.2,100.0


In [5]:
def psi(old_labels, new_labels):
    old = pd.Series(old_labels).value_counts(normalize=True)
    new = pd.Series(new_labels).value_counts(normalize=True)
    idx = old.index.union(new.index)
    old = old.reindex(idx, fill_value=0).clip(lower=1e-6)   # avoid ln(0)
    new = new.reindex(idx, fill_value=0).clip(lower=1e-6)
    return round(float(((new - old) * np.log(new / old)).sum()), 4)

edges = np.quantile(train_score, np.linspace(0, 1, 11))   # train deciles
edges[0], edges[-1] = -np.inf, np.inf
band = lambda s: pd.cut(s, edges)

kaggle_score, kaggle_pd = scorecard_score(kaggle)
print("PSI train → test  :", psi(band(train_score), band(test_score)))
print("PSI train → kaggle:", psi(band(train_score), band(kaggle_score)))
print(f"\nMean score: train {train_score.mean():.0f}, test {test_score.mean():.0f}, kaggle {kaggle_score.mean():.0f}")


PSI train → test  : 0.0002
PSI train → kaggle: 0.0015

Mean score: train 568, test 568, kaggle 568


In [8]:
def bins_for(df, f):
    """Bin NUMBER (0, 1, 2, ...) for ONE feature – much lighter than text labels."""
    return bp.get_binned_variable(f).transform(df[f].values, metric="indices")

csi = pd.Series({f: psi(bins_for(train, f), bins_for(kaggle, f)) for f in lr["features"]},
                name="CSI")
csi.sort_values(ascending=False).to_frame()


,CSI
EXT_SOURCE_1,0.0647
AMT_CREDIT_TO_GOODS_RATIO,0.0308
BUREAU_YEARS_SINCE_FIRST,0.0238
PREV_GRANTED_TO_REQUESTED,0.0154
PREV_REFUSAL_RATE,0.0109
EXT_SOURCE_2,0.0093
EXT_SOURCE_3,0.0076
BUREAU_DEBT_TO_CREDIT,0.0067
INST_LATE_SHARE_LAST12,0.0034
YEARS_EMPLOYED,0.0029


In [9]:
print(f"EXT_SOURCE_1 missing: train {train['EXT_SOURCE_1'].isna().mean():.1%}, "
      f"kaggle {kaggle['EXT_SOURCE_1'].isna().mean():.1%}")


EXT_SOURCE_1 missing: train 56.3%, kaggle 42.1%
